# **FINDING OUT SINGAPORE ETHOS**

## **1. Title and Overview**
### Ethical Alignment Layer for Singapore Policy Modeling

This notebook implements a practical "ethical alignment" layer for evaluating
policy scenarios in Singapore. It assumes you already have (or will generate)
predicted outcomes for each policy scenario (e.g., GST options, SME grants).

**What you'll get:**
- A principled way to encode Singapore's governance ethos in code
- Scenario scoring against that ethos (with hard constraints)
- Sensitivity analysis over weights (because we don't "know" true weights)
- Optional: learn weights from national documents (Budget/Forward SG)
- Optional: estimate "revealed" weights from past decisions


## **Code: Imports & Utilities**

In [ ]:
# Core libs
import pandas as pd
import numpy as np
# Optional: for optimization in revealed preference example
try:
    from scipy.optimize import minimize
except Exception as e:
    print("Note: scipy not installed. Revealed-preference fitting will be skipped.")
    minimize = None

## **Example Scenarios (replace with your outputs)**

In [ ]:
# Example scenarios with predicted outcomes from your upstream models.
# Replace with your DataFrame. Each row is a policy scenario.
scenarios = pd.DataFrame([
    {"scenario":"A_GST_8",
     "gdp_growth":2.1,                 # % yoy
     "unemp_rate":3.1,                 # %
     "gini_delta":+0.004,              # change in Gini (post-policy - pre)
     "emissions_delta":-1.0,           # % change
     "public_trust_idx":0.2,           # 0..1 proxy from NLP sentiment
     "debt_gdp_10y":39.0,              # projected % of GDP in 10y
     "bottom30_real_income_delta":+0.6 # % change
    },
    {"scenario":"B_GST_9_rebates",
     "gdp_growth":1.9,
     "unemp_rate":3.2,
     "gini_delta":+0.001,
     "emissions_delta":-1.2,
     "public_trust_idx":0.4,
     "debt_gdp_10y":38.5,
     "bottom30_real_income_delta":+1.1
    },
    {"scenario":"C_SME_grants",
     "gdp_growth":2.4,
     "unemp_rate":3.0,
     "gini_delta":-0.002,
     "emissions_delta":-0.4,
     "public_trust_idx":0.1,
     "debt_gdp_10y":40.2,
     "bottom30_real_income_delta":+0.3
    },
])
scenarios


## **Code: Encode Principles & Presets**

In [ ]:
# Principles = Singapore's governance ethos distilled into measurable buckets.
# Each principle lists metrics with a desired direction and a "target band"
# used for normalization (mapping raw metric -> score in [0,1]).

principles = {
    "Prosperity_Stability": {  # growth, jobs, competitiveness
        "weight": 0.35,
        "metrics": {
            "gdp_growth": {"direction": "+", "target": (0.0, 3.5)},  # % yoy
            "unemp_rate": {"direction": "-", "target": (2.0, 4.5)},  # %
        }
    },
    "Social_Cohesion_Equity": {  # safety nets, shared progress
        "weight": 0.30,
        "metrics": {
            "gini_delta": {"direction": "-", "target": (-0.01, 0.01)},
            "bottom30_real_income_delta": {"direction": "+", "target": (0.0, 2.0)},
        }
    },
    "Intergenerational_Prudence": {  # fiscal sustainability
        "weight": 0.20,
        "metrics": {
            "debt_gdp_10y": {"direction": "-", "target": (35.0, 45.0)},
        }
    },
    "Environmental_Stewardship": {  # Green Plan alignment
        "weight": 0.10,
        "metrics": {
            "emissions_delta": {"direction": "-", "target": (-2.0, 0.0)},
        }
    },
    "Trust_and_Legitimacy": {  # public acceptance / implementation risk
        "weight": 0.05,
        "metrics": {
            "public_trust_idx": {"direction": "+", "target": (0.0, 1.0)},
        }
    },
}

# Weight presets. You can switch between them to see how rankings move.
presets = {
    "baseline_gov": {k: v["weight"] for k, v in principles.items()},
    "equity_tilted": {
        "Prosperity_Stability": 0.25,
        "Social_Cohesion_Equity": 0.40,
        "Intergenerational_Prudence": 0.15,
        "Environmental_Stewardship": 0.15,
        "Trust_and_Legitimacy": 0.05,
    },
    "growth_tilted": {
        "Prosperity_Stability": 0.45,
        "Social_Cohesion_Equity": 0.20,
        "Intergenerational_Prudence": 0.20,
        "Environmental_Stewardship": 0.10,
        "Trust_and_Legitimacy": 0.05,
    },
}


## **Normalization Helper**

In [ ]:
def normalize(value, lo, hi, direction):
    """
    Map a raw metric value into [0, 1] using a policy-relevant band [lo, hi].
    direction: '+' means higher is better; '-' means lower is better.
    """
    if hi == lo:
        return 0.0
    if direction == "+":
        score = (value - lo) / (hi - lo)
    else:
        score = (hi - value) / (hi - lo)
    return float(np.clip(score, 0.0, 1.0))


## **Hard Constraints (Deontological “red lines”)**

In [ ]:
HARD_CONSTRAINTS = [
    lambda row: row["bottom30_real_income_delta"] >= 0.0,  # no net harm to bottom 30%
    lambda row: row["unemp_rate"] <= 5.0,                  # avoid severe job deterioration
]

## **Scoring Engine**

In [ ]:
def score_scenario(row, principle_weights=None):
    """
    Compute per-principle scores and overall alignment for one scenario (Series).
    Returns: feasible (bool), overall_score (float), principle_scores (dict)
    """
    if principle_weights is None:
        principle_weights = {k: v["weight"] for k, v in principles.items()}

    # Feasibility check
    feasible = all(rule(row) for rule in HARD_CONSTRAINTS)

    # Per-principle scores
    principle_scores = {}
    for pname, pdef in principles.items():
        pweight = principle_weights.get(pname, pdef["weight"])
        subs = []
        for metric, spec in pdef["metrics"].items():
            subs.append(
                normalize(
                    row[metric],
                    spec["target"][0],
                    spec["target"][1],
                    spec["direction"],
                )
            )
        principle_scores[pname] = (float(np.mean(subs)), pweight)

    # Weighted aggregation
    overall = sum(s * w for (s, w) in principle_scores.values())
    return feasible, overall, {k: s for k, (s, _) in principle_scores.items()}

def score_table(df, preset="baseline_gov", custom_weights=None):
    """
    Score all scenarios and return a ranked table.
    - preset: key in `presets`
    - custom_weights: dict overriding weights (optional)
    """
    weights = custom_weights or presets.get(preset, presets["baseline_gov"])
    rows = []
    for _, r in df.iterrows():
        feasible, overall, pbreak = score_scenario(r, weights)
        rows.append({
            "scenario": r["scenario"],
            "feasible": feasible,
            "overall_alignment": round(overall, 3),
            **{f"{k}_score": round(v, 3) for k, v in pbreak.items()},
        })
    return pd.DataFrame(rows).sort_values(
        ["feasible", "overall_alignment"], ascending=[False, False]
    )


## **Rank Scenarios Under Different Presets**

In [ ]:
for preset in presets.keys():
    print(f"\n=== Preset: {preset} ===")
    display(score_table(scenarios, preset=preset))


## **Weight Sensitivity Sweep**

In [ ]:
def sweep_weights(base_weights, span=0.2, n=200, random_state=42):
    """
    Randomly perturb weights around a base set (Gaussian noise),
    clamp to >=0, then renormalize to sum=1.
    """
    rng = np.random.default_rng(random_state)
    keys = list(base_weights.keys())
    draws = []
    for _ in range(n):
        w = {k: max(0.0, rng.normal(base_weights[k], span * max(base_weights[k], 1e-6))) for k in keys}
        total = sum(w.values()) or 1.0
        w = {k: v / total for k, v in w.items()}
        draws.append(w)
    return draws

def ranking_stability(df, base_weights, n=200):
    winners = []
    for w in sweep_weights(base_weights, n=n):
        table = score_table(df, custom_weights=w)
        winners.append(table.iloc[0]["scenario"])
    from collections import Counter
    return Counter(winners)

print("Stability under baseline_gov weights:")
ranking_stability(scenarios, presets["baseline_gov"], n=500)


## **Explain One Scenario (Why did it score?)**

In [ ]:
def explain_scenario(row, preset="baseline_gov", custom_weights=None):
    weights = custom_weights or presets[preset]
    details = []
    for pname, pdef in principles.items():
        w = weights[pname]
        subs = []
        for metric, spec in pdef["metrics"].items():
            s = normalize(row[metric], spec["target"][0], spec["target"][1], spec["direction"])
            subs.append((metric, s, row[metric], spec))
        pscore = float(np.mean([s for _, s, _, _ in subs]))
        details.append({
            "principle": pname,
            "weight": round(w, 3),
            "principle_score": round(pscore, 3),
            "metric_breakdown": [
                {
                    "metric": m,
                    "score": round(s, 3),
                    "value": v,
                    "direction": spec["direction"],
                    "target_band": spec["target"],
                }
                for (m, s, v, spec) in subs
            ],
        })
    overall = sum(d["principle_score"] * (custom_weights.get(d["principle"], presets[preset][d["principle"]]) if custom_weights else d["weight"]) for d in details)
    return round(overall, 3), details

# Example: explain the second scenario under baseline
overall, breakdown = explain_scenario(scenarios.iloc[1], preset="baseline_gov")
overall, breakdown[:2]  # preview first two principles


## Document-Informed Weights (light NLP)

We approximate the "current ethos" by scanning recent national documents
(e.g., Forward Singapore, Budget speeches). We count keyword families per
principle to produce a salience-based weight vector. This is a simplistic baseline
you can later upgrade to embeddings.


## **Derive Weights from Policy Texts (simple)**

In [ ]:
import re
from collections import Counter

keyword_families = {
    "Prosperity_Stability": ["growth","competitiveness","productivity","jobs","inflation","resilience"],
    "Social_Cohesion_Equity": ["cohesion","equity","inclusive","workfare","support","inequality","vulnerable"],
    "Intergenerational_Prudence": ["reserves","sustainability","fiscal","future","prudence","stewardship"],
    "Environmental_Stewardship": ["green","emissions","climate","sustainability","net zero","environment"],
    "Trust_and_Legitimacy": ["trust","confidence","consensus","social compact","partnership"],
}

def derive_weights_from_text(doc_text):
    text = re.sub(r"[^a-zA-Z\s]", " ", doc_text.lower())
    tokens = text.split()
    cnt = Counter(tokens)
    raw = {p: sum(cnt[w] for w in words) for p, words in keyword_families.items()}
    total = sum(raw.values()) or 1.0
    weights = {p: (v / total) for p, v in raw.items()}
    return weights

# EXAMPLE (replace with your own loaded documents)
example_docs = """
Our priorities are growth, productivity, and jobs. We will support vulnerable groups,
strengthen social compact, and ensure sustainability for the future. Trust and resilience matter.
"""
learned = derive_weights_from_text(example_docs)

# Blend with equal weights (alpha controls reliance on doc)
alpha = 0.6
equal = {k: 1/len(principles) for k in principles}
blended = {k: alpha*learned.get(k, 0.0) + (1-alpha)*equal[k] for k in principles}
s = sum(blended.values()) or 1.0
blended = {k: v/s for k, v in blended.items()}
blended


## Revealed-Preference Weights

If you can reconstruct past decisions where multiple options were considered,
you can estimate "implicit" weights that make the chosen option score highest
vs. alternatives. Below is a constrained least-squares style sketch.

**Data needed:**
- For each decision `d`, the principle scores `S_{d,p}` for the chosen option,
  and for the best plausible alternative.


## **Revealed Weights (sketch)**

In [ ]:
def fit_weights_revealed(X_chosen, X_alt):
    """
    X_chosen: (D x P) matrix of principle scores for chosen options across D decisions
    X_alt:    (D x P) matrix of scores for best alternative per decision
    Returns weights w (>=0, sum=1) that maximize average margin (X_chosen - X_alt) @ w.
    """
    if minimize is None:
        raise RuntimeError("scipy not available; install scipy to run this")

    P = X_chosen.shape[1]

    def obj(w):
        margin = (X_chosen @ w) - (X_alt @ w)  # want this large
        return -margin.mean() + 0.1 * np.square(w).sum()  # L2 regularization

    constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]
    bounds = [(0.0, 1.0)] * P
    w0 = np.ones(P) / P
    res = minimize(obj, w0, bounds=bounds, constraints=constraints, method="SLSQP")
    return res.x, res

# Dummy example (3 decisions, 5 principles) — replace with real data
Xc = np.array([[0.7,0.5,0.6,0.4,0.5],
               [0.8,0.4,0.5,0.6,0.5],
               [0.6,0.6,0.7,0.3,0.6]])
Xa = np.array([[0.6,0.6,0.6,0.5,0.5],
               [0.7,0.5,0.6,0.5,0.4],
               [0.5,0.7,0.6,0.4,0.5]])

if minimize is not None:
    w_hat, res = fit_weights_revealed(Xc, Xa)
    revealed_weights = {p: float(w) for p, w in zip(principles.keys(), w_hat)}
    revealed_weights


## **Swap Presets & Rerun Rankings**
## Using learned weights

You can slot any weight dict into `score_table` via `custom_weights=...`.
Try: `blended` (doc-informed) or `revealed_weights` (if fitted).

Also: always report *stability* (fraction of random weight draws where each
scenario ranks #1). If rankings flip a lot, the choice is value-sensitive.


### Run with Custom Weights + Stability

In [ ]:
# Run with doc-informed blended weights
doc_table = score_table(scenarios, custom_weights=blended)
display(doc_table)

# Stability around the blended weights
stability = ranking_stability(scenarios, blended, n=500)
stability


## Notes & Next Steps

- **Replace example metrics** with your model's outputs (ensure column names match).
- **Tune target bands** per metric using Singapore-specific context (historical ranges, policy targets).
- **Refine constraints** to reflect red lines (e.g., poverty gap must not widen).
- Add a small UI (Streamlit) with sliders for:
  - Policy parameters (e.g., GST %, rebate size)
  - Weight presets (baseline/equity/growth/doc-informed)
  - Show: ranked table, per-principle bars, radar profiles, and explanation accordions.
- Document everything:
  - Data sources (data.gov.sg, MAS, MTI, etc.)
  - Assumptions and limitations
  - Versioning of weights/targets (MLflow or a simple YAML)
